# M1 — improvement attempt: description + affected products (#7)

**Question:** does adding metadata already in the database (FR-23 allows *description and metadata*) improve the baseline (macro-F1 0.516)?

**Method, without cheating on the test set:**

| Set | Period | Use |
|---|---|---|
| fit | 2018-01 → 2024-06 | train the candidate models |
| validation | 2024-07 → 2025-06 | **choose** the best model and its settings |
| test | 2025-07 → 2026-10 | evaluated **once**, at the end, with the chosen model (same test set as the baseline) |

Choosing settings on the test set would make the result too optimistic: the test set only judges the final choice.

## 0. Settings

In [1]:
TRAIN_FROM = "2018-01-01"
VALID_FROM = "2024-07-01"
TEST_FROM = "2025-07-01"     # same test period as the baseline notebook
RANDOM_STATE = 42
TARGET_MACRO_F1 = 0.70
C_VALUES = [0.5, 1.0, 2.0]    # regularization strengths tried on the validation set

## 1. Connection to the operational database

In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg

def find_repo_root(start=Path.cwd()):
    for folder in [start, *start.parents]:
        if (folder / "docker-compose.yml").exists():
            return folder
    raise FileNotFoundError("Run the notebook from inside the threatlens repository")

def read_env(path):
    values = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            values[key.strip()] = value.strip()
    return values

ROOT = find_repo_root()
env = read_env(ROOT / ".env")
conn = psycopg.connect(host="localhost", port=int(env.get("POSTGRES_PORT", 5432)),
                       dbname=env["POSTGRES_DB"], user=env["POSTGRES_USER"], password=env["POSTGRES_PASSWORD"])

def query(sql, params=None):
    with conn.cursor() as cur:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])

print("Connected to", env["POSTGRES_DB"])

Connected to threatlens


## 2. Data: same labels as the baseline + affected products

Label = CVSS 3.x severity (CNA first, else CISA ADP). New feature: the affected vendors and products published by the CNA, e.g. `v=fortinet`, `p=fortinet/fortios`.

In [3]:
raw = query("""
    SELECT v.cve_id, v.description, v.published_at, m.provider, m.version, m.severity
    FROM cti_vulnerability v
    JOIN cti_cvssmetric m ON m.vulnerability_id = v.id
    WHERE m.version IN ('3.0', '3.1') AND m.provider IN ('CNA', 'CISA_ADP')
""")
priority = {("CNA", "3.1"): 0, ("CNA", "3.0"): 1, ("CISA_ADP", "3.1"): 2, ("CISA_ADP", "3.0"): 3}
raw["rank"] = [priority[(p, v)] for p, v in zip(raw.provider, raw.version)]
data = (raw.sort_values("rank").drop_duplicates("cve_id")
           .query("severity != 'NONE'").dropna(subset=["published_at"]).reset_index(drop=True))
data["published_at"] = pd.to_datetime(data.published_at, utc=True)
data = data[data.published_at >= pd.Timestamp(TRAIN_FROM, tz="UTC")]

products = query("""
    SELECT v.cve_id, string_agg(DISTINCT p.vendor || '/' || p.name, '|') AS products
    FROM cti_affectedproduct a
    JOIN cti_product p ON p.id = a.product_id
    JOIN cti_vulnerability v ON v.id = a.vulnerability_id
    GROUP BY v.cve_id
""")
data = data.merge(products, on="cve_id", how="left")
data["products"] = data.products.fillna("")
print(f"{len(data):,} labelled CVEs; {(data.products != '').mean():.1%} have at least one affected product")

202,325 labelled CVEs; 84.8% have at least one affected product


In [4]:
valid_from, test_from = pd.Timestamp(VALID_FROM, tz="UTC"), pd.Timestamp(TEST_FROM, tz="UTC")
fit = data[data.published_at < valid_from]
valid = data[(data.published_at >= valid_from) & (data.published_at < test_from)]
train_full = data[data.published_at < test_from]     # fit + validation, used for the final model
test = data[data.published_at >= test_from]
CLASSES = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
pd.DataFrame({"CVEs": [len(fit), len(valid), len(test)]}, index=["fit", "validation", "test"])

,CVEs
fit,75815
validation,38754
test,87756


## 3. Candidate models

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline

def product_tokens(value):
    """'fortinet/fortios|fortinet/fortiproxy' -> vendor and product tokens."""
    tokens = set()
    for item in filter(None, value.split("|")):
        tokens.add("v=" + item.split("/", 1)[0])
        tokens.add("p=" + item)
    return sorted(tokens)

def text_features():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=200_000,
                           sublinear_tf=True, strip_accents="unicode")

def make_model(use_products, C):
    columns = [("text", text_features(), "description")]
    if use_products:
        columns.append(("products", CountVectorizer(tokenizer=product_tokens, token_pattern=None,
                                                    lowercase=False, binary=True, min_df=5), "products"))
    return Pipeline([
        ("features", ColumnTransformer(columns)),
        ("classifier", LogisticRegression(C=C, max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
    ])

candidates = [("text only", False, 1.0)] + [("text + products", True, c) for c in C_VALUES]

## 4. Choice on the validation set (the test set is not used here)

In [6]:
rows = []
for name, use_products, C in candidates:
    model = make_model(use_products, C).fit(fit, fit.severity)
    pred = model.predict(valid)
    rows.append({"features": name, "C": C, "use_products": use_products,
                 "validation_macro_F1": round(f1_score(valid.severity, pred, average="macro"), 3)})
    print(f"{name:16} C={C:<4} validation macro-F1 = {rows[-1]['validation_macro_F1']}")
choice = pd.DataFrame(rows).sort_values("validation_macro_F1", ascending=False).reset_index(drop=True)
best = choice.iloc[0]
print("\nChosen:", best.features, "C =", best.C)
choice

text only        C=1.0  validation macro-F1 = 0.577
text + products  C=0.5  validation macro-F1 = 0.568
text + products  C=1.0  validation macro-F1 = 0.579
text + products  C=2.0  validation macro-F1 = 0.587

Chosen: text + products C = 2.0


,features,C,use_products,validation_macro_F1
0,text + products,2.0,True,0.587
1,text + products,1.0,True,0.579
2,text only,1.0,False,0.577
3,text + products,0.5,True,0.568


## 5. Final evaluation on the test set (once)

The chosen configuration is retrained on fit + validation (2018-01 → 2025-06), then evaluated on the same test period as the baseline.

In [7]:
final = make_model(bool(best.use_products), float(best.C)).fit(train_full, train_full.severity)
pred = final.predict(test)
final_f1 = round(f1_score(test.severity, pred, average="macro"), 3)

baseline_file = ROOT / "ml" / "models" / "m1_baseline_metrics.json"
baseline_f1 = json.loads(baseline_file.read_text())["macro_f1"] if baseline_file.exists() else None
print(f"Test macro-F1: {final_f1}  (baseline text only: {baseline_f1})\n")
print(classification_report(test.severity, pred, labels=CLASSES, digits=3))

Test macro-F1: 0.521  (baseline text only: 0.516)

              precision    recall  f1-score   support

         LOW      0.323     0.406     0.360      3971
      MEDIUM      0.686     0.636     0.660     36507
        HIGH      0.663     0.589     0.624     37837
    CRITICAL      0.356     0.578     0.440      9441

    accuracy                          0.599     87756
   macro avg      0.507     0.552     0.521     87756
weighted avg      0.624     0.599     0.607     87756



In [8]:
matrix = pd.DataFrame(confusion_matrix(test.severity, pred, labels=CLASSES),
                      index=[f"true {c}" for c in CLASSES], columns=[f"pred {c}" for c in CLASSES])
# Severity classes are ordered: share of predictions that are right or one class away
order = {c: i for i, c in enumerate(CLASSES)}
within_one = np.mean([abs(order[t] - order[p]) <= 1 for t, p in zip(test.severity, pred)])
print(f"Right or one class away: {within_one:.1%}")
matrix

Right or one class away: 93.5%


,pred LOW,pred MEDIUM,pred HIGH,pred CRITICAL
true LOW,1611,1542,584,234
true MEDIUM,2461,23222,7753,3071
true HIGH,799,8179,22272,6587
true CRITICAL,109,899,2976,5457


## 6. Result and decision

In [9]:
from datetime import date

decision = "GO" if final_f1 >= TARGET_MACRO_F1 else "GO with reservations" if final_f1 >= 0.60 else "NO-GO"
summary = {
    "date": date.today().isoformat(),
    "features": best.features, "C": float(best.C),
    "fit_period": f"{TRAIN_FROM} to {VALID_FROM}", "validation_period": f"{VALID_FROM} to {TEST_FROM}",
    "test_period": f"from {TEST_FROM}", "test_size": int(len(test)),
    "macro_f1": float(final_f1), "baseline_macro_f1": baseline_f1,
    "right_or_one_class_away": round(float(within_one), 3),
    "decision": decision,
}
(ROOT / "ml" / "models" / "m1_improvement_metrics.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

{
  "date": "2026-10-20",
  "features": "text + products",
  "C": 2.0,
  "fit_period": "2018-01-01 to 2024-07-01",
  "validation_period": "2024-07-01 to 2025-07-01",
  "test_period": "from 2025-07-01",
  "test_size": 87756,
  "macro_f1": 0.521,
  "baseline_macro_f1": 0.516,
  "right_or_one_class_away": 0.935,
  "decision": "NO-GO"
}


**Interpretation.** Adding the affected vendors and products to the description gives a negligible gain: test macro-F1 **0.521** vs **0.516** for text only (same test period, Jul 2025 – Oct 2026, 87,756 CVEs). The configuration was chosen on the validation set (text + products, C = 2: 0.587 vs 0.577 for text only), the test set was used once.

- **Temporal drift:** macro-F1 drops from 0.587 (validation, Jul 2024 – Jun 2025) to 0.521 (test, Jul 2025 – Oct 2026): recent CVEs are harder to predict, so M1 must be retrained regularly (model versions, FR-26).
- **Errors stay close:** 93.5 % of predictions are right or one class away; only 109 CRITICAL CVEs are predicted LOW.

**Decision unchanged: NO-GO for a text-only M1, risk R4 plan applied.** Sprint 3 levers: the CNA that published the CVE and the CWE (both require loading new fields), and treating severity as an ordered target.

In [ ]:
conn.close()